In [1]:
!pip install -q transformers torch accelerate

In [2]:
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [3]:
MODEL_NAME = "ProsusAI/finbert"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)

model.eval()

print(device)

config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  438MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

cuda


In [4]:
import torch
import torch.nn.functional as F

labels = ["positive", "negative", "neutral"]


def finbert_batch_predict(texts, batch_size=32):

    predictions = []

    positive_probs = []
    negative_probs = []
    neutral_probs = []

    for i in range(0, len(texts), batch_size):

        batch = texts[i:i + batch_size]

        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():

            outputs = model(**encoded)

            probs = F.softmax(outputs.logits, dim=1)

        for row in probs.cpu().numpy():

            positive = row[0]
            negative = row[1]
            neutral = row[2]

            label = labels[row.argmax()]

            positive_probs.append(positive)
            negative_probs.append(negative)
            neutral_probs.append(neutral)

            predictions.append(label)

    return (
        predictions,
        positive_probs,
        negative_probs,
        neutral_probs
    )

In [6]:
news_df = pd.read_csv("/news_clean.csv")

In [7]:
preds, pos, neg, neu = finbert_batch_predict(
    news_df["Headline"].tolist(),
    batch_size=32
)

news_df["Sentiment"] = preds

news_df["Positive_Prob"] = pos

news_df["Negative_Prob"] = neg

news_df["Neutral_Prob"] = neu

In [8]:
score_map = {
    "positive": 1,
    "neutral": 0,
    "negative": -1
}

news_df["Sentiment_Score"] = news_df["Sentiment"].map(score_map)

In [9]:
daily_sentiment = (
    news_df
    .groupby(["Date", "Ticker"])
    .agg(
        Avg_Sentiment=("Sentiment_Score", "mean"),
        Sentiment_STD=("Sentiment_Score", "std"),
        News_Count=("Headline", "count"),

        Positive_Ratio=("Positive_Prob", "mean"),
        Neutral_Ratio=("Neutral_Prob", "mean"),
        Negative_Ratio=("Negative_Prob", "mean")
    )
    .reset_index()
)

daily_sentiment["Sentiment_STD"] = (
    daily_sentiment["Sentiment_STD"]
    .fillna(0)
)

In [10]:
print(daily_sentiment.shape)

daily_sentiment.head(10)

(1755, 8)


,Date,Ticker,Avg_Sentiment,Sentiment_STD,News_Count,Positive_Ratio,Neutral_Ratio,Negative_Ratio
0,2011-03-03,NVDA,-1.00,0.000000,1,0.015090,0.159814,0.825096
1,2011-03-07,NVDA,0.00,0.000000,2,0.064805,0.909439,0.025756
2,2011-03-08,NVDA,0.25,0.957427,4,0.331308,0.374511,0.294181
3,2011-03-09,NVDA,0.00,0.000000,3,0.060045,0.922541,0.017414
4,2011-03-11,NVDA,0.00,0.000000,2,0.070367,0.888172,0.041461
5,2011-03-15,NVDA,0.00,0.000000,2,0.045974,0.940028,0.013998
6,2011-03-16,NVDA,-0.50,0.577350,4,0.021000,0.509191,0.469809
7,2011-03-23,NVDA,0.00,0.000000,1,0.030990,0.900871,0.068139
8,2011-03-24,NVDA,0.00,0.000000,1,0.033382,0.947051,0.019567
9,2011-03-25,NVDA,0.00,0.000000,2,0.055075,0.909770,0.035155


In [11]:
daily_sentiment.to_csv(
    "daily_sentiment.csv",
    index=False
)

print("daily_sentiment.csv saved successfully!")

daily_sentiment.csv saved successfully!
